In [2]:
import numpy as np
import pickle
import os
import h5py

import matplotlib.pyplot as plt
from util import (
    get_intensity,
    get_phase,
    re_im_combined,
    change_domain_and_adjust_energy,
)


def save_figure(save_name, save_format, save_dir):
    if save_name is None:
        raise ValueError("Save name is not specified")
    if "." + save_format not in save_name:
        save_name += "." + save_format
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    plt.savefig(
        os.path.join(save_dir, save_name),
        bbox_inches="tight",
        dpi=300,
        transparent=True,
        format=save_format,
    )


def intensity_phase_plot(
    domains,
    fields,
    labels,
    colors,
    domain_type,
    xlims=None,
    y_label=None,
    normalize=False,
    offsets=None,
    save_format="jpg",
    save_name=None,
    plot_show=True,
    plot_hold=False,
    save_dir="",
    save=False,
    axs=None,
    fig=None,
):
    """
    Plot intensity and phase of a field
    """
    if domain_type == "time":
        factor = 1e12
        x_label = "time (ps)"
    elif domain_type == "wavelength":
        factor = 1e9
        x_label = "wavelength (nm)"
    elif domain_type == "frequency" or domain_type == "freq":
        factor = 1e-12
        x_label = "frequency (THz)"

    # NOTE: Modifying the domains itself will change the original domains in the calling scope!
    # Which was causing our problems
    factored_domains = [domain * factor for domain in domains]

    intensities = [get_intensity(field) for field in fields]
    phases = [np.unwrap(get_phase(field)) for field in fields]

    y_label_2 = "Phase (rad)"

    if axs is None:
        fig, axs = plt.subplots(nrows=1, ncols=1, figsize=(6, 4), clear=False)

    if normalize:
        y_label_1 = "Norm. Intensity (a.u.)"
        intensities = [intensity / np.max(intensity) for intensity in intensities]
    else:
        y_label_1 = "Fluence (J/m^2)"

    axs.set_xlabel(x_label)

    for i, intensity in enumerate(intensities):
        if y_label is None:
            y_label_1 = y_label_1
        else:
            y_label_1 = y_label

        if offsets is not None:
            offset = offsets[i]
            
        axs.plot(
            factored_domains[i],
            intensity + offset,
            color=colors[i],
            label=labels[i],
            alpha=0.6,
        )

    axs.legend()

    if xlims is not None:
        axs.set_xlim(xlims[0], xlims[1])

    axs.set_ylabel(y_label_1, color="black")
    axs2 = axs.twinx()

    for i in range(len(phases)):
        axs2.plot(
            factored_domains[i],
            phases[i],
            color=colors[i],
            linestyle="dashed",
            alpha=0.6,
        )

    axs2.set_ylabel(y_label_2, color="black")

    if save:
        save_figure(save_name, save_format, save_dir)

    return fig


def plot_a_bunch_of_fields(
    freq_vectors_sfg_list,
    fields_sfg_list,
    freq_vectors_shg1_list,
    fields_shg1_list,
    freq_vectors_shg2_list,
    fields_shg2_list,
    sfg_time_vector_list,
    sfg_freq_to_time_list,
    shg1_time_vector_list,
    shg1_freq_to_time_list,
    shg2_time_vector_list,
    shg2_freq_to_time_list,
    labels_list,
    colors_list,
    model_save_name,
    fig_save_dir,
    file_save_name=None,
    normalize=False,
):
    nrows = 2
    ncols = 3

    plt.figure()

    new_fig, new_axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(30, 15))

    # Flatten the array of axes if it's 2D
    if nrows > 1 and ncols > 1:
        new_axs = new_axs.flatten()
    else:  # 1D array of axes
        new_axs = new_axs.reshape(-1)

    print("------- True vs Prediction Frequency Domain --------")
    print("*** SFG ***")
    fig_pfg4 = intensity_phase_plot(
        freq_vectors_sfg_list,
        fields_sfg_list,
        labels_list,
        colors_list,
        "freq",
        normalize=normalize,
        offsets=[0, 0],
        save_format="jpg",
        save_name=model_save_name + "_pfg4.jpg",
        plot_show=True,
        plot_hold=False,
        save_dir=fig_save_dir,
        axs=new_axs[0],
    )

    print("*** SHG1 ***")
    fig_pfg5 = intensity_phase_plot(
        freq_vectors_shg1_list,
        fields_shg1_list,
        labels_list,
        colors_list,
        "freq",
        normalize=normalize,
        offsets=[0, 0],
        save_format="jpg",
        save_name=model_save_name + "_pfg5.jpg",
        plot_show=True,
        plot_hold=False,
        save_dir=fig_save_dir,
        axs=new_axs[1],
    )

    print("*** SHG2 ***")
    fig_pfg6 = intensity_phase_plot(
        freq_vectors_shg2_list,
        fields_shg2_list,
        labels_list,
        colors_list,
        "freq",
        normalize=normalize,
        offsets=[0, 0],
        save_format="jpg",
        save_name=model_save_name + "_pfg6.jpg",
        plot_show=True,
        plot_hold=False,
        save_dir=fig_save_dir,
        axs=new_axs[2],
    )

    print("------- True vs Prediction Time Domain --------")

    print("*** SFG ***")
    fig_ptd4 = intensity_phase_plot(
        sfg_time_vector_list,
        sfg_freq_to_time_list,
        labels_list,
        colors_list,
        "time",
        xlims=[-15, 15],
        normalize=normalize,
        offsets=[0, 0],
        save_format="jpg",
        save_name=model_save_name + "_ptd4.jpg",
        plot_show=True,
        plot_hold=False,
        save_dir=fig_save_dir,
        axs=new_axs[3],
    )

    print("*** SHG1 ***")
    fig_ptd5 = intensity_phase_plot(
        shg1_time_vector_list,
        shg1_freq_to_time_list,
        labels_list,
        colors_list,
        "time",
        normalize=normalize,
        offsets=[0, 0],
        save_format="jpg",
        save_name=model_save_name + "_ptd5.jpg",
        plot_show=True,
        plot_hold=False,
        save_dir=fig_save_dir,
        axs=new_axs[4],
    )

    print("*** SHG2 ***")
    fig_ptd6 = intensity_phase_plot(
        shg2_time_vector_list,
        shg2_freq_to_time_list,
        labels_list,
        colors_list,
        "time",
        normalize=normalize,
        offsets=[0, 0],
        save_format="jpg",
        save_name=model_save_name + "_ptd6.jpg",
        plot_show=True,
        plot_hold=False,
        save_dir=fig_save_dir,
        axs=new_axs[5],
    )

    new_fig.tight_layout()
    plt.show()

    file_save_name = (
        file_save_name
        or model_save_name + f"_All_{'normalized' if normalize else 'orig'}.jpg"
    )

    save_figure(file_save_name, "jpg", fig_save_dir)



In [4]:
freq_vectors_shg1 = np.load("../Data/shg_freq_domain_ds.npy")
freq_vectors_shg2 = freq_vectors_shg1  # these are equivalent here
freq_vectors_sfg = np.load("../Data/sfg_freq_domain_ds.npy")

domain_spacing_1 = (
    freq_vectors_shg1[1] - freq_vectors_shg1[0]
)  # * 1e12  # scaled to be back in Hz
domain_spacing_2 = freq_vectors_shg2[1] - freq_vectors_shg2[0]  # * 1e12
domain_spacing_3 = freq_vectors_sfg[1] - freq_vectors_sfg[0]  # * 1e12

factors_freq = {
    "beam_area": 400e-6**2 * np.pi,
    "grid_spacing": [domain_spacing_1, domain_spacing_2, domain_spacing_3],
    "domain_spacing_1": domain_spacing_1,
    "domain_spacing_2": domain_spacing_2,
    "domain_spacing_3": domain_spacing_3,
}  # beam radius 400 um (and circular beam)

sfg_original_freq = np.load("../Data/sfg_original_freq_vector.npy")
sfg_original_time = np.load("../Data/sfg_original_time_vector.npy")


In [8]:
sfg_original_time_ds = sfg_original_time[1] - sfg_original_time[0]
print(sfg_original_time_ds*10**(15))
sfg_original_freq_ds = sfg_original_freq[1] - sfg_original_freq[0]
print(sfg_original_freq_ds*10**(-12))

16.50000000003815
0.0018495501893945312


In [9]:
.12/0.0018

66.66666666666667

In [10]:
80/16.5

4.848484848484849